In [ ]:
import torchvision
import torchvision.transforms as transforms
import torch
import torch.nn as nn
import numpy as np

import ModelFxp12 as fxp

In [ ]:
class GetChannel:
    def __init__(self, channel):
      self.channel = channel

    def __call__(self, img):
        return  torch.unsqueeze(img[self.channel], 0)

stats = ((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(*stats),
    #transforms.Grayscale()
    GetChannel(1)
])

# Create Datasets
testset = torchvision.datasets.CIFAR10(root='CIFAR-DATA', train = False, download=True, transform=transform)

# Load Datasets
test_loader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=True)

100%|██████████| 170498071/170498071 [00:02<00:00, 72191402.43it/s]


Extracting CIFAR-DATA/cifar-10-python.tar.gz to CIFAR-DATA


In [ ]:
class NetCIFAR10(nn.Module):
    def __init__(self):
        nn.Module.__init__(self)

        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 26, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(26, 26, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # output: 26, 16, 16
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(26, 26, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(26, 26, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # output: 26, 8, 8
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(26, 26, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(26, 26, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # output: 26, 4, 4
        )

        self.fc1 = nn.Sequential(
            nn.Flatten(),
            nn.Linear(26 * 4 * 4, 10)
        )

    def forward(self, xb):
        xb = self.conv1(xb)
        #print(xb.shape)
        xb = self.conv2(xb)
        #print(xb.shape)
        xb = self.conv3(xb)
        #print(xb.shape)
        xb = self.fc1(xb)

        return xb

In [ ]:
# Computes the average accuracy
def evaluate_fxp(model, val_loader):

    batch_accuracies = []   # Accuracies of each forwarded batch

    for images, labels in val_loader:
        outputs = model(images)                    # Generates batch predictions

        # Computes the batch accuracy
        _, preds = torch.max(torch.from_numpy(outputs), dim=1)
        accuracy = torch.tensor(torch.sum(preds == labels).item() / len(preds))

        # Append the batch accuracy
        batch_accuracies.append(accuracy)

    # Computes the epoch average accuracy
    accuracy = torch.stack(batch_accuracies).mean() # Average accuracy

    return accuracy

In [ ]:
# Create the fixed point model
def model_fxp(model, shift):

    layers = [
        # conv1
        fxp.Conv(model.conv1[0], shift, shift),
        fxp.ReLU(),
        fxp.Conv(model.conv1[2], shift, shift),
        fxp.ReLU(),
        fxp.MaxPool2d(2),

        # conv2
        fxp.Conv(model.conv2[0], shift, shift),
        fxp.ReLU(),
        fxp.Conv(model.conv2[2], shift, shift),
        fxp.ReLU(),
        fxp.MaxPool2d(2),

        # conv3
        fxp.Conv(model.conv3[0], shift, shift),
        fxp.ReLU(),
        fxp.Conv(model.conv3[2], shift, shift),
        fxp.ReLU(),
        fxp.MaxPool2d(2),

        # fc1
        fxp.Linear(model.fc1[1], shift, shift)
    ]

    return fxp.ModelFxp(layers)

In [ ]:
# Load the quantized floating point model
model = NetCIFAR10()
model.load_state_dict(torch.load('train2_qat_sh6-8187.pt', map_location=torch.device('cpu')))


# Report (GetChannel(1))
for shift in (range(6,10)):
    model_test = model_fxp(model, shift)
    print(f"Shift: {shift}, accuracy: {evaluate_fxp(model_test, test_loader)}")
    model_test.AccumulatorsWidth()
    model_test.WeightsWidth()
    print('\n')

#   shift   accuracy
#


layers[0]: Conv
layers[1]: ReLU
layers[2]: Conv
layers[3]: ReLU
layers[4]: MaxPool2d
layers[5]: Conv
layers[6]: ReLU
layers[7]: Conv
layers[8]: ReLU
layers[9]: MaxPool2d
layers[10]: Conv
layers[11]: ReLU
layers[12]: Conv
layers[13]: ReLU
layers[14]: MaxPool2d
layers[15]: Linear
Shift: 6, accuracy: 0.8003362417221069
ConvLayers[0]: min = -360, max = 403 MIN_WIDTH = 10.0
ConvLayers[1]: min = -837, max = 613 MIN_WIDTH = 11.0
ConvLayers[2]: min = -1287, max = 978 MIN_WIDTH = 12.0
ConvLayers[3]: min = -1077, max = 964 MIN_WIDTH = 12.0
ConvLayers[4]: min = -1302, max = 953 MIN_WIDTH = 12.0
ConvLayers[5]: min = -1431, max = 1366 MIN_WIDTH = 12.0
FullLayers[0]: min = -1026, max = 2560 MIN_WIDTH = 13.0
ConvLayers[0] min width: 3.0
ConvLayers[1] min width: 3.0
ConvLayers[2] min width: 3.0
ConvLayers[3] min width: 3.0
ConvLayers[4] min width: 3.0
ConvLayers[5] min width: 3.0
FullLayers[0] min width: 3.0


layers[0]: Conv
layers[1]: ReLU
layers[2]: Conv
layers[3]: ReLU
layers[4]: MaxPool2d
layers[

In [ ]:
# Load the quantized floating point model
model = NetCIFAR10()
model.load_state_dict(torch.load('train2_qat_sh7-8186.pt', map_location=torch.device('cpu')))


# Report (GetChannel(1))
for shift in (range(7,10)):
    model_test = model_fxp(model, shift)
    print(f"Shift: {shift}, accuracy: {evaluate_fxp(model_test, test_loader)}")
    model_test.AccumulatorsWidth()
    model_test.WeightsWidth()
    print('\n')

#   shift   accuracy
#


layers[0]: Conv
layers[1]: ReLU
layers[2]: Conv
layers[3]: ReLU
layers[4]: MaxPool2d
layers[5]: Conv
layers[6]: ReLU
layers[7]: Conv
layers[8]: ReLU
layers[9]: MaxPool2d
layers[10]: Conv
layers[11]: ReLU
layers[12]: Conv
layers[13]: ReLU
layers[14]: MaxPool2d
layers[15]: Linear
Shift: 7, accuracy: 0.8163567781448364
ConvLayers[0]: min = -727, max = 807 MIN_WIDTH = 11.0
ConvLayers[1]: min = -1730, max = 1228 MIN_WIDTH = 12.0
ConvLayers[2]: min = -2607, max = 1976 MIN_WIDTH = 13.0
ConvLayers[3]: min = -2226, max = 1939 MIN_WIDTH = 13.0
ConvLayers[4]: min = -2779, max = 1989 MIN_WIDTH = 13.0
ConvLayers[5]: min = -3044, max = 2872 MIN_WIDTH = 13.0
FullLayers[0]: min = -2119, max = 5459 MIN_WIDTH = 14.0
ConvLayers[0] min width: 3.0
ConvLayers[1] min width: 3.0
ConvLayers[2] min width: 3.0
ConvLayers[3] min width: 3.0
ConvLayers[4] min width: 3.0
ConvLayers[5] min width: 3.0
FullLayers[0] min width: 3.0


layers[0]: Conv
layers[1]: ReLU
layers[2]: Conv
layers[3]: ReLU
layers[4]: MaxPool2d
la

In [ ]:
# Load the quantized floating point model
model = NetCIFAR10()
model.load_state_dict(torch.load('train2_qat_sh6-8184.pt', map_location=torch.device('cpu')))


# Report (GetChannel(1))
for shift in (range(6,10)):
    model_test = model_fxp(model, shift)
    print(f"Shift: {shift}, accuracy: {evaluate_fxp(model_test, test_loader)}")
    model_test.AccumulatorsWidth()
    model_test.WeightsWidth()
    print('\n')

#   shift   accuracy
#


layers[0]: Conv
layers[1]: ReLU
layers[2]: Conv
layers[3]: ReLU
layers[4]: MaxPool2d
layers[5]: Conv
layers[6]: ReLU
layers[7]: Conv
layers[8]: ReLU
layers[9]: MaxPool2d
layers[10]: Conv
layers[11]: ReLU
layers[12]: Conv
layers[13]: ReLU
layers[14]: MaxPool2d
layers[15]: Linear
Shift: 6, accuracy: 0.8004351258277893
ConvLayers[0]: min = -361, max = 404 MIN_WIDTH = 10.0
ConvLayers[1]: min = -834, max = 609 MIN_WIDTH = 11.0
ConvLayers[2]: min = -1284, max = 976 MIN_WIDTH = 12.0
ConvLayers[3]: min = -1078, max = 967 MIN_WIDTH = 12.0
ConvLayers[4]: min = -1306, max = 955 MIN_WIDTH = 12.0
ConvLayers[5]: min = -1442, max = 1367 MIN_WIDTH = 12.0
FullLayers[0]: min = -1030, max = 2587 MIN_WIDTH = 13.0
ConvLayers[0] min width: 3.0
ConvLayers[1] min width: 3.0
ConvLayers[2] min width: 3.0
ConvLayers[3] min width: 3.0
ConvLayers[4] min width: 3.0
ConvLayers[5] min width: 3.0
FullLayers[0] min width: 3.0


layers[0]: Conv
layers[1]: ReLU
layers[2]: Conv
layers[3]: ReLU
layers[4]: MaxPool2d
layers[